In [6]:
import jdatetime
import datetime
import pandas as pd

# ===============================
# 🔹 Gregorian date range
# ===============================
start_date = datetime.date(2012, 3, 20)
end_date   = datetime.date(2030, 12, 29)

# List to store all generated records
records = []

# ===============================
# 🔹 Function to build one DimDate record for a given day
# ===============================
def build_dimdate_record(miladi):
    # Convert Gregorian → Jalali
    jalali = jdatetime.date.fromgregorian(date=miladi)

    year = jalali.year
    month = jalali.month
    day = jalali.day

    # DateKey structure: YYYYMMDD in Jalali
    date_key = int(f"{year}{month:02}{day:02}")
    shamsi_date = f"{year}/{month:02}/{day:02}"

    # Semester (1 or 2)
    semester = 1 if month <= 6 else 2
    semester_name = "نيمسال اول" if semester == 1 else "نيمسال دوم"

    # Quarter calculation based on Jalali month
    if month <= 3:
        quarter = 1
        quarter_name = "بهار"
    elif month <= 6:
        quarter = 2
        quarter_name = "تابستان"
    elif month <= 9:
        quarter = 3
        quarter_name = "پاييز"
    else:
        quarter = 4
        quarter_name = "زمستان"

    # Jalali month names
    month_names = ["فروردين","ارديبهشت","خرداد","تير","مرداد","شهريور",
                   "مهر","آبان","آذر","دي","بهمن","اسفند"]
    month_name = month_names[month - 1]

    # Persian weekday names
    weekday_names = ["دوشنبه","سه‌شنبه","چهارشنبه","پنجشنبه","جمعه","شنبه","يكشنبه"]
    day_name = weekday_names[miladi.weekday()]

    # Day of year (Gregorian)
    day_of_year = (miladi - datetime.date(miladi.year, 1, 1)).days + 1

    # Day of Quarter
    q_start_month = {1:1, 2:4, 3:7, 4:10}[quarter]
    quarter_start = jdatetime.date(year, q_start_month, 1).togregorian()
    day_of_quarter = (miladi - quarter_start).days + 1

    # Day of Semester
    s_start_month = 1 if semester == 1 else 7
    semester_start = jdatetime.date(year, s_start_month, 1).togregorian()
    day_of_semester = (miladi - semester_start).days + 1

    # ISO Week of Year
    week_of_year = miladi.isocalendar()[1]

    # Week of Month
    first_day_month = jdatetime.date(year, month, 1).togregorian()
    week_of_month = week_of_year - first_day_month.isocalendar()[1] + 1

    # Week of Quarter & Week of Semester
    week_of_quarter = (day_of_quarter - 1) // 7 + 1
    week_of_semester = (day_of_semester - 1) // 7 + 1

    # YearMonth fields
    year_month = int(f"{year}{month:02}")
    year_month_name = f"ماه {month_name} سال {year}"

    return {
        "DateKey": date_key,
        "MiladiDate": miladi.strftime("%Y-%m-%d"),
        "Date": shamsi_date,
        "CalendarYear": year,
        "Semester": semester,
        "Quarter": quarter,
        "MonthNo": month,
        "WeekOfMonth": week_of_month,
        "WeekOfQuarter": week_of_quarter,
        "WeekOfSemester": week_of_semester,
        "WeekOfYear": week_of_year,
        "DayOfWeek": miladi.weekday() + 1,
        "DayOfMonth": day,
        "DayOfQuarter": day_of_quarter,
        "DayOfSemester": day_of_semester,
        "DayOfYear": day_of_year,
        "DayName": day_name,
        "MonthName": month_name,
        "QuarterName": quarter_name,
        "SemesterName": semester_name,
        "YearMonth": year_month,
        "YearMonthName": year_month_name,
        "SaleInvoice": 0,
        "SaleTarget": 0,
        "Treasury": 0
    }

# ===============================
# 🔹 Loop through all dates in the range
# ===============================
current = start_date
while current <= end_date:
    records.append(build_dimdate_record(current))
    current += datetime.timedelta(days=1)

# Convert to DataFrame
df = pd.DataFrame(records)

# ===============================
# 🔹 Save output to Excel file
# ===============================
output_file = "DimDate_2012_2030.xlsx"
df.to_excel(output_file, index=False)

print("📁 File generated →", output_file)
print("Total rows:", len(df))


📁 File generated → DimDate_2012_2030.xlsx
Total rows: 6859
